<a href="https://colab.research.google.com/github/edgarcia2022-a11y/Hystrix/blob/main/Ed_Garcia_ML_Basics_House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Kaggle Data
# Source: https://www.kaggle.com/datasets/abdulwadood11220/usa-house-sales-data?resource=download

df = pd.read_csv('us_house_Sales_data.csv')
print(df.columns)

# Clean 'Price' column and convert to numeric
df['Price'] = df['Price'].replace({r'[$,]': ''}, regex=True).astype(float)

# Clean 'Area (Sqft)' column and convert to numeric
df['Area (Sqft)'] = df['Area (Sqft)'].replace({r' sqft': ''}, regex=True).astype(float)

# Features and target
X = df[['Area (Sqft)', 'City']]
y = df['Price']

# Preprocessing: One-hot encode the City column and pass through Area (Sqft)
preprocessor = ColumnTransformer(
transformers=[
('City', OneHotEncoder(sparse_output=False), ['City'])
], remainder='passthrough')

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
('preprocessor', preprocessor),
('regressor', LinearRegression())
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

# Train model
model.fit(X_train, y_train)

# Make prediction for a new house: 2000 sq ft in Sacramento
# Note: 'Downtown' is not a city in the dataset, using 'Sacramento' as an example city
new_house = pd.DataFrame({'Area (Sqft)': [2000], 'City': ['Sacramento']})
predicted_price = model.predict(new_house);
print(f"Predicted price for a 2000 sq ft house in Sacramento with 2000 Area (Sqft): ${predicted_price[0]:.2f}")

# Display model coefficients
# Ensure feature names match the preprocessor output
feature_names = (
    model.named_steps['preprocessor']
    .named_transformers_['City']
    .get_feature_names_out(['City'])
    .tolist()
    + ['Area (Sqft)'] # 'Area (Sqft)' is passed through directly, so its name remains
)
coefficients = model.named_steps['regressor'].coef_
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

Index(['Price', 'Address', 'City', 'Zipcode', 'State', 'Bedrooms', 'Bathrooms',
       'Area (Sqft)', 'Lot Size', 'Year Built', 'Days on Market',
       'Property Type', 'MLS ID', 'Listing Agent', 'Status', 'Listing URL'],
      dtype='object')
Predicted price for a 2000 sq ft house in Sacramento with 2000 Area (Sqft): $811777.89

Model Coefficients:
City_Fresno: 17640.38
City_Los Angeles: 7741.22
City_Sacramento: -7750.17
City_San Diego: -947.34
City_San Francisco: -16684.09
Area (Sqft): -18.42
